# Deep Dive: Chaining Prompts for Agentic Reasoning

## 📋 Summary

**Prompt chaining** is the practice of breaking a complex task into a sequence of smaller,
focused prompts where the **output of each step becomes the input for the next**.

Instead of one giant prompt that tries to do everything, a chain has:
- **Specialized prompts** — each optimized for one job
- **Data flow** — structured output passed between steps
- **Gate checks** — validation between stages to catch errors early
- **Branching logic** — conditional routing based on intermediate results

### Why chain prompts?
| Single prompt | Prompt chain |
|---|---|
| Easier to write | Easier to debug (one step at a time) |
| Can drift or mix concerns | Each step is focused and testable |
| Hard to retry a single step | Can retry just the failing stage |
| Unstable for complex tasks | Reliable for multi-step workflows |

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Single prompt vs. simple chain | 🟢 Beginner |
| 2 | Structured data between steps | 🟢 Beginner |
| 3 | Gate checks & validation | 🟡 Intermediate |
| 4 | Branching — conditional routing | 🟡 Intermediate |
| 5 | Fan-out / fan-in pattern | 🔴 Advanced |
| 6 | Exercises | 🏋️ Practice |


## 0. Setup

In [1]:
# key input
import os
from google.colab import userdata
# set the key on colab on the left "key" icon (SECRETS)
# Automatically configures the key for SDKs to grab behind the scenes
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [2]:
# Install / import
from openai import OpenAI
from IPython.display import Markdown, display
import json, re

# ── API key ──────────────────────────────────────────────────────────────────

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"] ,
)

MODEL = "gpt-4.1-nano"

def chat(system: str, user: str, temperature: float = 0.3) -> str:
    """Single-turn helper. Returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content

def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))
    return Markdown(f"**{label}**\n\n{text}\n\n---").data

def chat_json(system: str, user: str) -> dict:
    """Like chat() but returns parsed JSON."""
    raw = chat(system, user, temperature=0.1)
    clean = raw.strip().strip('`').replace('json\n','',1).strip()
    return json.loads(clean)

## 1. Single Prompt vs. Simple Chain 🟢

**Task:** Analyze a customer review, extract its sentiment & key issues, then
draft a personalized customer service reply.

We'll do it in one giant prompt first, then break it into a chain.


In [3]:
REVIEW = """
I've been using this blender for 3 weeks. The motor is incredibly powerful and it blends
smoothies perfectly — no chunks at all. But the lid is a disaster. It popped off twice while
running at high speed, spraying smoothie all over my kitchen. Customer service took 5 days
to respond and just sent me a generic troubleshooting guide. Very disappointed overall.
"""

# ── Single prompt approach ────────────────────────────────────────────────────
single_system = """You are a customer experience specialist.
Given a customer review, do ALL of the following in one response:
1. Extract the sentiment (positive/negative/mixed)
2. List the key issues the customer mentions
3. Draft a professional, empathetic reply that addresses each issue
Format: sentiment first, then issues, then reply."""

r_single = chat(single_system, f"Review:\n{REVIEW}")
show("Single prompt — all in one", r_single)


**Single prompt — all in one**

Sentiment: Negative

Issues:
1. Lid popping off during use, causing messes
2. Slow and unhelpful customer service response
3. Overall disappointment with the product experience

Reply:
Dear Customer,

Thank you for sharing your feedback. We're glad to hear that you are satisfied with the motor power and blending performance of our blender. However, we sincerely apologize for the inconvenience caused by the lid issue and the delay in our customer service response. Your safety and satisfaction are our top priorities, and we understand how frustrating it must be to experience the lid popping off and the resulting mess. We are actively working to improve our product design and support process. Please contact us directly at [contact information], and we will prioritize assisting you with a replacement lid or a suitable solution. Thank you for your patience and for giving us the opportunity to make this right.

---

"**Single prompt — all in one**\n\nSentiment: Negative\n\nIssues:\n1. Lid popping off during use, causing messes\n2. Slow and unhelpful customer service response\n3. Overall disappointment with the product experience\n\nReply:\nDear Customer,\n\nThank you for sharing your feedback. We're glad to hear that you are satisfied with the motor power and blending performance of our blender. However, we sincerely apologize for the inconvenience caused by the lid issue and the delay in our customer service response. Your safety and satisfaction are our top priorities, and we understand how frustrating it must be to experience the lid popping off and the resulting mess. We are actively working to improve our product design and support process. Please contact us directly at [contact information], and we will prioritize assisting you with a replacement lid or a suitable solution. Thank you for your patience and for giving us the opportunity to make this right.\n\n---"

In [4]:
# ── Chain approach ───────────────────────────────────────────────────────────

# Step 1: Extract structured data from the review
step1_system = """You are a sentiment analysis engine.
Analyze the review and return ONLY valid JSON with this structure:
{
  "overall_sentiment": "positive|negative|mixed",
  "positives": ["..."],
  "negatives": ["..."],
  "customer_emotion": "frustrated|disappointed|happy|angry|neutral"
}"""

step1_result = chat_json(step1_system, f"Review:\n{REVIEW}")
print("✅ Step 1 — Extracted data:")
print(json.dumps(step1_result, indent=2))


✅ Step 1 — Extracted data:
{
  "overall_sentiment": "mixed",
  "positives": [
    "powerful motor",
    "blends smoothies perfectly"
  ],
  "negatives": [
    "lid popped off twice",
    "sprayed smoothie all over kitchen",
    "slow customer service",
    "disappointed overall"
  ],
  "customer_emotion": "disappointed"
}


In [5]:
# Step 2: Draft the reply using Step 1's structured output
step2_system = """You are a customer service specialist writing on behalf of a blender company.
You are given structured analysis of a customer review. Write a professional, warm, specific reply.
- Acknowledge each negative point directly
- Mention what you will do to resolve it
- Thank them for the positives
- Keep it under 150 words
- Do NOT use generic phrases like 'We value your feedback'"""

step2_input = f"""Sentiment analysis of the review:
{json.dumps(step1_result, indent=2)}

Original review:
{REVIEW}"""

step2_result = chat(step2_system, step2_input)
show("✅ Step 2 — Drafted reply", step2_result)


**✅ Step 2 — Drafted reply**

Thank you for sharing your experience. We're glad the powerful motor and smooth blending meet your expectations. However, we're sorry about the lid popping off and the spray mess—that's certainly frustrating. We will review the lid design and follow up with you directly to provide a replacement or solution. We also apologize for the slow customer service response; we’re working to improve our response times. Your feedback helps us improve. Thank you for your honesty, and we appreciate your patience as we resolve these issues.

---

"**✅ Step 2 — Drafted reply**\n\nThank you for sharing your experience. We're glad the powerful motor and smooth blending meet your expectations. However, we're sorry about the lid popping off and the spray mess—that's certainly frustrating. We will review the lid design and follow up with you directly to provide a replacement or solution. We also apologize for the slow customer service response; we’re working to improve our response times. Your feedback helps us improve. Thank you for your honesty, and we appreciate your patience as we resolve these issues.\n\n---"

### 🧐 Comparison
- Which approach gave a more targeted, specific reply?
- In the chain version, could you reuse Step 1's output for other purposes (e.g., a dashboard)?


## 2. Structured Data Between Steps 🟢

Good chains use **structured intermediate outputs** (JSON) so each step knows
exactly what format to expect. Let's build a 3-step content pipeline.

**Task:** Turn a raw news article snippet into a formatted newsletter entry.


In [6]:
ARTICLE = """
Scientists at MIT announced a breakthrough in room-temperature superconductivity on Thursday.
The team demonstrated a new material — a hydrogen-rich compound under modest pressure —
that conducts electricity without resistance at 25°C. If confirmed, this could revolutionize
power grids, electric motors, and quantum computers. The work was published in Nature.
Lead researcher Dr. Elena Vasquez said the team spent 7 years on the discovery.
"""

# Step 1: Extract key facts
facts_system = """Extract key facts from the article as JSON:
{
  "headline_topic": "...",
  "key_finding": "...",
  "who": ["..."],
  "where": "...",
  "potential_impact": ["..."],
  "source_publication": "...",
  "time_frame": "..."
}"""

facts = chat_json(facts_system, f"Article:\n{ARTICLE}")
print("Step 1 — Facts extracted:")
print(json.dumps(facts, indent=2))


Step 1 — Facts extracted:
{
  "headline_topic": "Room-temperature superconductivity breakthrough",
  "key_finding": "A hydrogen-rich compound under modest pressure conducts electricity without resistance at 25\u00b0C",
  "who": [
    "Scientists at MIT",
    "Lead researcher Dr. Elena Vasquez"
  ],
  "where": "MIT",
  "potential_impact": [
    "Revolutionize power grids",
    "Transform electric motors",
    "Advance quantum computers"
  ],
  "source_publication": "Nature",
  "time_frame": "Thursday (recently announced, work spanned 7 years)"
}


In [7]:
# Step 2: Score newsworthiness / choose angle
angle_system = """You are a newsletter editor. Given article facts, decide:
{
  "angle": "breakthrough|skeptical|explainer|human_interest",
  "target_audience": "general|technical|business",
  "suggested_headline": "...",
  "tone": "excited|neutral|cautious"
}"""

angle = chat_json(angle_system, json.dumps(facts))
print("Step 2 — Editorial angle:")
print(json.dumps(angle, indent=2))


Step 2 — Editorial angle:
{
  "angle": "breakthrough",
  "target_audience": "general",
  "suggested_headline": "MIT Scientists Achieve Room-Temperature Superconductivity in Hydrogen-Rich Compound",
  "tone": "excited"
}


In [8]:
# Step 3: Write the newsletter entry using both previous outputs
write_system = """You are a science newsletter writer.
Write a 100-word newsletter blurb using the provided facts and editorial decisions.
Match the tone and angle exactly. End with one sentence about why this matters to readers."""

write_input = f"""Facts: {json.dumps(facts)}
Editorial: {json.dumps(angle)}"""

final_entry = chat(write_system, write_input, temperature=0.5)
show("Step 3 — Final newsletter entry", final_entry)


**Step 3 — Final newsletter entry**

MIT scientists, led by Dr. Elena Vasquez, have achieved a groundbreaking breakthrough in superconductivity. After seven years of dedicated research, they announced in Nature that a hydrogen-rich compound conducts electricity without resistance at a remarkable 25°C under modest pressure—room temperature! This revolutionary discovery could transform power grids, making energy transmission vastly more efficient, and propel advancements in electric motors and quantum computing. Such a major leap in materials science promises to reshape our technological landscape. For everyday readers, this breakthrough brings us closer to a future where super-efficient energy and cutting-edge electronics are within reach.

---

'**Step 3 — Final newsletter entry**\n\nMIT scientists, led by Dr. Elena Vasquez, have achieved a groundbreaking breakthrough in superconductivity. After seven years of dedicated research, they announced in Nature that a hydrogen-rich compound conducts electricity without resistance at a remarkable 25°C under modest pressure—room temperature! This revolutionary discovery could transform power grids, making energy transmission vastly more efficient, and propel advancements in electric motors and quantum computing. Such a major leap in materials science promises to reshape our technological landscape. For everyday readers, this breakthrough brings us closer to a future where super-efficient energy and cutting-edge electronics are within reach.\n\n---'

## 3. Gate Checks & Validation 🟡

A **gate check** is a validation step between two chain stages.
It prevents garbage from propagating downstream.

Three types of gate checks:
1. **Format gate** — is the JSON valid and complete?
2. **Logic gate** — does the content make sense?
3. **LLM gate** — ask a model to assess quality before continuing


In [9]:
# Utility: format gate
def format_gate(data: dict, required_keys: list[str]) -> tuple[bool, str]:
    """Check that all required keys are present and non-empty."""
    missing = [k for k in required_keys if not data.get(k)]
    if missing:
        return False, f"Missing or empty keys: {missing}"
    return True, "OK"

# Utility: LLM gate — asks the model to evaluate quality
def llm_gate(content: str, criteria: str) -> tuple[bool, str]:
    """Use an LLM to decide if content meets criteria. Returns (pass, reason)."""
    gate_system = """You are a strict quality checker. Evaluate the content against the criteria.
Respond with ONLY valid JSON: {"pass": true/false, "reason": "one sentence"}"""
    result = chat_json(gate_system, f"Criteria: {criteria}\n\nContent:\n{content}")
    return result.get("pass", False), result.get("reason", "unknown")


# ── Demo: 3-step chain WITH gate checks ─────────────────────────────────────
def analyze_review_with_gates(review: str) -> dict:
    print("=== Starting chain with gate checks ===\n")

    # Step 1
    step1_result = chat_json(step1_system, f"Review:\n{review}")

    # Gate 1: format check
    ok, reason = format_gate(step1_result, ["overall_sentiment", "negatives", "customer_emotion"])
    print(f"Gate 1 (format): {'✅ PASS' if ok else '❌ FAIL'} — {reason}")
    if not ok:
        return {"error": "Stage 1 format gate failed", "detail": reason}

    # Gate 2: logic check — does a negative review have negatives listed?
    if step1_result["overall_sentiment"] == "negative" and not step1_result["negatives"]:
        print("Gate 2 (logic): ❌ FAIL — Negative sentiment but no negatives listed")
        return {"error": "Logic inconsistency"}
    print("Gate 2 (logic): ✅ PASS")

    # Step 2
    step2_input = f"Sentiment analysis:\n{json.dumps(step1_result)}\n\nOriginal review:\n{review}"
    reply = chat(step2_system, step2_input)

    # Gate 3: LLM quality gate
    ok3, reason3 = llm_gate(
        reply,
        "The reply must be specific (mention at least one concrete issue from the review), "
        "empathetic, and under 150 words. It must NOT contain the phrase 'We value your feedback'."
    )
    print(f"Gate 3 (LLM quality): {'✅ PASS' if ok3 else '❌ FAIL'} — {reason3}")

    if not ok3:
        # Retry with stronger instructions
        print("  → Retrying step 2 with stricter prompt...")
        stronger_system = step2_system + "\nIMPORTANT: Be very specific. Mention exact product issues."
        reply = chat(stronger_system, step2_input)

    return {"analysis": step1_result, "reply": reply}

result = analyze_review_with_gates(REVIEW)
print("\n" + "="*50)
show("Final gated reply", result.get("reply", "Error"))


=== Starting chain with gate checks ===

Gate 1 (format): ✅ PASS — OK
Gate 2 (logic): ✅ PASS
Gate 3 (LLM quality): ✅ PASS — The response is specific about issues with the lid popping off and slow customer service, empathetic in tone, and under 150 words.



**Final gated reply**

Thank you for sharing your experience. We're glad you enjoy the powerful motor and smooth blends. However, we sincerely apologize for the lid popping off and the spray mess—that's certainly frustrating. We will review the lid design to prevent this issue and will reach out to you directly to provide a replacement or solution. We're also sorry for the slow response from our customer service; we're working to improve our response times. Your feedback helps us improve, and we appreciate your honesty. Thank you for giving our blender a try.

---

"**Final gated reply**\n\nThank you for sharing your experience. We're glad you enjoy the powerful motor and smooth blends. However, we sincerely apologize for the lid popping off and the spray mess—that's certainly frustrating. We will review the lid design to prevent this issue and will reach out to you directly to provide a replacement or solution. We're also sorry for the slow response from our customer service; we're working to improve our response times. Your feedback helps us improve, and we appreciate your honesty. Thank you for giving our blender a try.\n\n---"

## 4. Branching — Conditional Routing 🟡

Real chains don't always go in a straight line. Based on intermediate results,
the chain routes to different next steps.

**Task:** Process a support ticket — route to the right department based on the issue type.


In [10]:
TICKETS = [
    "My payment was charged twice but I only received one order. Please refund the extra charge.",
    "I can't log in to my account. I've tried resetting my password 3 times and nothing works.",
    "The product I received is broken. The screen has a crack right out of the box.",
    "I'd like to know when you're getting the blue version back in stock.",
]

# Step 1: Classify the ticket
classifier_system = """Classify a support ticket and return JSON:
{
  "category": "billing|technical|product_defect|inventory",
  "priority": "high|medium|low",
  "summary": "one sentence summary"
}
High priority = customer lost money or is completely blocked."""

# Step 2 prompts — one per department
DEPARTMENT_PROMPTS = {
    "billing": """You are a billing specialist. Write a concise reply acknowledging the financial
issue, confirming you will investigate, and providing a timeline (24 hours for review).""",
    "technical": """You are a technical support engineer. Write a step-by-step troubleshooting
reply. Be specific. Provide 2-3 concrete steps to try.""",
    "product_defect": """You are a returns specialist. Write a reply that apologizes for the defect,
immediately offers a replacement or refund, and explains the process.""",
    "inventory": """You are a customer service rep. Write a friendly reply explaining you've noted
the interest, and offer to notify them when the item is back.""",
}

def process_ticket(ticket: str) -> dict:
    # Classify
    classification = chat_json(classifier_system, f"Ticket:\n{ticket}")
    category = classification.get("category", "technical")

    # Route to the right department prompt
    department_system = DEPARTMENT_PROMPTS.get(category, DEPARTMENT_PROMPTS["technical"])

    # Generate reply
    reply = chat(department_system, f"Support ticket:\n{ticket}")

    return {
        "ticket": ticket[:80] + "...",
        "category": category,
        "priority": classification.get("priority"),
        "reply": reply,
    }

# Process all tickets
for i, ticket in enumerate(TICKETS, 1):
    result = process_ticket(ticket)
    print(f"Ticket {i}: [{result['category'].upper()}] [{result['priority'].upper()}]")
    show(f"Ticket {i} Reply", result["reply"])


Ticket 1: [BILLING] [HIGH]


**Ticket 1 Reply**

Thank you for bringing this to our attention. I will investigate the issue and review your account. You can expect a response within 24 hours. We appreciate your patience.

---

Ticket 2: [TECHNICAL] [HIGH]


**Ticket 2 Reply**

Thank you for reaching out. Let's try the following steps to resolve the login issue:

1. **Clear Browser Cache and Cookies:**  
   - If you're using a web browser, go to your browser settings and clear your cache and cookies. This can resolve issues caused by stored data interfering with the login process.

2. **Try Logging in from a Different Browser or Incognito Mode:**  
   - Open a new incognito/private browsing window and attempt to log in again. Alternatively, try using a different browser to see if the issue persists.

3. **Verify Your Account Status:**  
   - Ensure your account hasn't been deactivated or restricted. If possible, check your email for any notifications from us regarding account status or contact our support team directly to verify your account.

Please try these steps and let me know if you're still unable to log in.

---

Ticket 3: [PRODUCT_DEFECT] [HIGH]


**Ticket 3 Reply**

Dear [Customer Name],

Thank you for reaching out to us. I sincerely apologize for the inconvenience caused by receiving a defective product with a cracked screen. This is certainly not the experience we want for our valued customers.

To resolve this promptly, we can offer you a replacement or a full refund—whichever you prefer. Please let us know your choice, and we will initiate the process immediately.

Here’s what to expect next:
1. If you choose a replacement, we will send out a new item right away and provide you with a prepaid return label to send back the defective product.
2. If you prefer a refund, we will process it as soon as we receive the returned item, or if you wish to keep the product, we can issue a refund without the need for return.

Please confirm your preferred resolution and provide any additional details or photos of the defect, if possible. Once we receive your response, we will expedite the process to ensure your satisfaction.

Again, I apologize for the inconvenience and appreciate your understanding. We look forward to resolving this for you quickly.

Best regards,  
[Your Name]  
Customer Support Team

---

Ticket 4: [INVENTORY] [MEDIUM]


**Ticket 4 Reply**

Hello!

Thank you for reaching out and for your interest in the blue version. I’ve noted your request and will be happy to notify you as soon as it’s back in stock. 

Please feel free to reach out if you have any other questions in the meantime. We appreciate your patience and look forward to assisting you soon!

Best regards,  
[Your Name]  
Customer Service Team

---

## 5. Fan-Out / Fan-In Pattern 🔴

**Fan-out**: one input → multiple parallel analyses (run the same prompt multiple times or
different specialized prompts in parallel)
**Fan-in**: combine all outputs into one synthesis

This is powerful for:
- Getting multiple perspectives on a decision
- Running the same extraction N times and aggregating for reliability
- Parallel processing of different aspects


In [11]:
# We'll analyze a business decision from multiple angles (fan-out)
# then synthesize them (fan-in)

DECISION = """
A startup is deciding whether to launch a premium subscription tier ($29/month)
alongside their free product. They have 50,000 free users, 12% monthly active rate,
$800K runway, and no revenue yet.
"""

ANALYST_PROMPTS = {
    "Financial Analyst": """You are a financial analyst. Analyze the decision purely from
a revenue and runway perspective. Be concise (3-4 sentences).""",

    "Product Manager": """You are a product manager. Analyze the decision from a user
experience and product-market fit perspective. Be concise (3-4 sentences).""",

    "Growth Marketer": """You are a growth marketing expert. Analyze the decision from a
user acquisition and conversion perspective. Be concise (3-4 sentences).""",

    "Risk Analyst": """You are a risk analyst. Identify the top 3 risks of this decision
and their likelihood. Be concise.""",
}

# Fan-out: run all analysts in sequence (parallel would use async/threading)
print("=== Fan-Out Phase ===")
perspectives = {}
for analyst, system in ANALYST_PROMPTS.items():
    response = chat(system, f"Decision to analyze:\n{DECISION}", temperature=0.4)
    perspectives[analyst] = response
    print(f"✅ {analyst} done")

# Fan-in: synthesize all perspectives
print("\n=== Fan-In Phase ===")
synthesis_input = f"Business decision:\n{DECISION}\n\nPerspectives gathered:\n"
for analyst, view in perspectives.items():
    synthesis_input += f"\n**{analyst}:**\n{view}\n"

synthesizer_system = """You are a senior business advisor.
You have gathered multiple expert perspectives on a business decision.
Write a 200-word executive summary that:
1. Highlights the most important insight from each perspective
2. Identifies where analysts agree and where they conflict
3. Gives a clear recommendation with the single biggest caveat"""

synthesis = chat(synthesizer_system, synthesis_input, temperature=0.3)
show("🔀 Fan-In — Executive Synthesis", synthesis)


=== Fan-Out Phase ===
✅ Financial Analyst done
✅ Product Manager done
✅ Growth Marketer done
✅ Risk Analyst done

=== Fan-In Phase ===


**🔀 Fan-In — Executive Synthesis**

The collective insights suggest cautious optimism regarding launching a $29/month premium tier. The Financial Analyst highlights that, given only 12% active users, immediate revenue gains are modest and unlikely to significantly extend the current $800K runway unless conversion rates improve substantially. The Product Manager emphasizes the importance of boosting user engagement and understanding demand before monetization, warning that premature premium offerings risk low uptake and user alienation. The Growth Marketer agrees that targeted efforts to increase engagement could enhance conversion prospects but notes that the current engagement level may not sustain profitable conversion rates. The Risk Analyst underscores the high likelihood of low conversion and market resistance, cautioning that revenue may remain insufficient regardless of the tier launch.

Consensus exists that improving user engagement should precede monetization efforts. Divergence lies in the perceived immediacy of revenue impact; analysts agree that the current metrics limit short-term gains but differ on whether the launch should be delayed or carefully tested.

**Recommendation:** Proceed with a cautious pilot of the premium tier, focusing on increasing engagement and testing willingness to pay. The biggest caveat is that without significant engagement growth, the premium launch may not materially improve financial stability.

---

'**🔀 Fan-In — Executive Synthesis**\n\nThe collective insights suggest cautious optimism regarding launching a $29/month premium tier. The Financial Analyst highlights that, given only 12% active users, immediate revenue gains are modest and unlikely to significantly extend the current $800K runway unless conversion rates improve substantially. The Product Manager emphasizes the importance of boosting user engagement and understanding demand before monetization, warning that premature premium offerings risk low uptake and user alienation. The Growth Marketer agrees that targeted efforts to increase engagement could enhance conversion prospects but notes that the current engagement level may not sustain profitable conversion rates. The Risk Analyst underscores the high likelihood of low conversion and market resistance, cautioning that revenue may remain insufficient regardless of the tier launch.\n\nConsensus exists that improving user engagement should precede monetization efforts. Di

## 6. Exercises 🏋️


### Exercise 1 — Build a 3-Step Resume Screener
Build a chain that:
1. Extracts structured info from a raw resume text (name, skills, years of experience)
2. Scores the candidate against a job description (0–100)
3. Drafts a personalized interview invitation or rejection email

Add a gate check: if score < 40, skip step 3 and return "not a fit".


In [ ]:
# Sample resume
RESUME = """
John Doe — 2 years experience as a junior developer.
Skills: Python basics, some JavaScript, HTML/CSS.
Education: Associate's degree in Information Technology.
Previous role: IT helpdesk at a local school district.
"""

JOB_DESCRIPTION = """
Senior Backend Engineer — 5+ years Python, deep knowledge of distributed systems,
experience with AWS, PostgreSQL, and microservices architecture required.
"""

# TODO: Build the 3-step chain
# Step 1: extract_candidate_info(resume) → dict
# Step 2: score_candidate(info, job_desc) → {score, reasoning}
# Gate: if score < 40, stop
# Step 3: draft_email(info, score) → str


### Exercise 2 — Add a Retry to a Gate
Extend the `analyze_review_with_gates` function so that if Gate 3 (LLM quality) fails,
it retries **up to 3 times** with progressively stricter instructions.
Log each retry attempt.


In [ ]:
# TODO: Modify analyze_review_with_gates to retry up to 3 times
def analyze_review_with_retry(review: str, max_retries: int = 3) -> dict:
    """Like analyze_review_with_gates but retries step 2 if quality gate fails."""
    pass  # TODO: implement


### Exercise 3 — Design Your Own Chain
Think of a real task in your work or studies that requires multiple steps.
Design a chain for it:
1. Draw the flow (even on paper) — what are the stages?
2. Define the input/output format for each stage
3. Identify where gate checks are needed
4. Implement at least the first 2 stages


In [ ]:
# TODO: Your chain idea here
# Describe the task:
task_description = """
[Describe the real-world task you want to chain]
"""

# Stage definitions:
stages = [
    {"name": "Stage 1", "input": "...", "output": "...", "prompt": "..."},
    # Add more stages
]


## 🏁 Summary & Key Takeaways

| Pattern | Use when |
|---|---|
| Simple chain | Task naturally decomposes into serial steps |
| Gate checks | Errors in one step would silently corrupt downstream steps |
| Branching | Different input types need different processing |
| Fan-out/fan-in | Multiple perspectives or parallel processing improves quality |

**Golden rules for chains:**
- Each step should do ONE thing well
- Always validate between steps
- Use structured JSON for data passing
- Design retry logic for critical gates

### What's next?
- **Lesson 5 Deep Dive**: LLM Feedback Loops — close the loop and let the model improve itself.
